In [1]:
import duckdb

con = duckdb.connect()
con.sql("CREATE VIEW t  AS SELECT * FROM '../data/raw/movimientos_dia_T.parquet'")
con.sql("CREATE VIEW t1 AS SELECT * FROM '../data/raw/movimientos_dia_T1.parquet'")

In [26]:
%load_ext sql
%sql con --alias duckdb
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = None

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


displaylimit: Value None will be treated as 0 (no limit)

In [3]:
%%sql
DESCRIBE t

column_name,column_type,null,key,default,extra
id_cliente,VARCHAR,YES,None,None,None
date,VARCHAR,YES,None,None,None
product,VARCHAR,YES,None,None,None
type,VARCHAR,YES,None,None,None
fund,VARCHAR,YES,None,None,None
amount,DOUBLE,YES,None,None,None
description,VARCHAR,YES,None,None,None
commercial_name,VARCHAR,YES,None,None,None


In [4]:
%%sql
DESCRIBE t1

column_name,column_type,null,key,default,extra
id_cliente,VARCHAR,YES,None,None,None
date,VARCHAR,YES,None,None,None
product,VARCHAR,YES,None,None,None
type,VARCHAR,YES,None,None,None
fund,VARCHAR,YES,None,None,None
amount,DOUBLE,YES,None,None,None
description,VARCHAR,YES,None,None,None
commercial_name,VARCHAR,YES,None,None,None


In [5]:
%%sql
SELECT * FROM t LIMIT 15

id_cliente,date,product,type,fund,amount,description,commercial_name
CLI001004,2024-09-22,Fondo de Inversión,out,Crecimiento,1490831.17,Retiro parcial,BBVA
CLI002933,2024-10-05,ETF,IN,Crecimiento,37940609.55,Comisión de administración,Fiduciaria Bogotá
CLI001379,2024-09-18,Fondo de Inversión,entrada,Mercado Monetario,36486859.6,Comisión de administración,Davivienda
CLI002534,2024-10-13,Bonos,salida,Renta Fija,49261090.81,Retiro parcial,Davivienda
CLI001495,2024-09-20,Bonos,entrada,Conservador,30456941.15,Retiro parcial,Itaú
CLI002622,2024-10-07,Fondo de Pensión,entrada,Internacional,40252484.99,Venta de activo,BBVA
CLI002941,2024-09-25,Divisas,salida,Conservador,12332128.76,Ajuste por valoración,BTG Pactual
CLI002391,2024-09-27,Acciones,entrada,Renta Fija,7642913.13,None,Corficolombiana
CLI002441,2024-09-29,CDT,salida,Renta Fija,13341992.67,None,Fiduciaria Bogotá
CLI001859,2024-09-15,Cuenta de Ahorro,entrada,Internacional,43526057.97,None,BTG Pactual


In [11]:
%%sql
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT id_cliente) AS ids_distintos,
       COUNT(*) - COUNT(id_cliente) AS ids_nulos
FROM t

filas,ids_distintos,ids_nulos
50000,3000,0


In [7]:
%%sql
SELECT count (*) FROM t

count_star()
50000


In [8]:
%%sql
SELECT count (*) FROM t1

count_star()
49000


In [ ]:
SELECT DISTINCT type FROM t

SyntaxError: invalid syntax (3710954133.py, line 1)

In [12]:
%%sql
SELECT * FROM t ORDER BY id_cliente, date LIMIT 20

id_cliente,date,product,type,fund,amount,description,commercial_name
CLI000001,2024-09-15,Divisas,entrada,Renta Fija,18473753.81,Reinversión automática,Corficolombiana
CLI000001,2024-09-15,Bonos,entrada,Internacional,2516265.71,Reinversión automática,Fiduciaria Bogotá
CLI000001,2024-09-17,Bonos,entrada,Internacional,6915867.08,Rendimiento del período,None
CLI000001,2024-09-18,Fondo de Pensión,salida,Internacional,1558625.61,Comisión de administración,Valores Bancolombia
CLI000001,2024-09-18,Bonos,salida,Internacional,31951275.58,Redención anticipada,Skandia
CLI000001,2024-09-20,ETF,salida,Conservador,29232177.84,Venta de activo,Itaú
CLI000001,2024-09-24,Fondo de Pensión,salida,Balanceado,28937945.84,Traslado entre fondos,None
CLI000001,2024-09-27,Acciones,salida,Renta Fija,41357514.56,Rendimiento del período,None
CLI000001,2024-09-27,Acciones,entrada,conservador,17729533.2,Rendimiento del período,Corficolombiana
CLI000001,2024-10-01,ETF,entrada,Mercado Monetario,46205433.83,Traslado entre fondos,BBVA


In [13]:
%%sql
SELECT
  COUNT(*) AS filas,
  COUNT(DISTINCT (id_cliente, date)) AS cliente_fecha,
  COUNT(DISTINCT (id_cliente, date, product)) AS mas_producto,
  COUNT(DISTINCT (id_cliente, date, product, fund)) AS mas_fondo,
  COUNT(DISTINCT (id_cliente, date, product, fund, type)) AS mas_tipo,
  COUNT(DISTINCT (id_cliente, date, product, fund, type, commercial_name)) AS mas_comercial,
  COUNT(DISTINCT (id_cliente, date, product, fund, type, commercial_name, amount, description)) AS fila_completa
FROM t

filas,cliente_fecha,mas_producto,mas_fondo,mas_tipo,mas_comercial,fila_completa
50000,40172,48586,49843,49933,49993,50000


In [14]:
%%sql
SELECT
  (SELECT COUNT(*) FROM t1) AS filas_t1,
  (SELECT COUNT(*) FROM (SELECT * FROM t INTERSECT ALL SELECT * FROM t1)) AS identicas,
  (SELECT COUNT(*) FROM (SELECT * FROM t EXCEPT ALL SELECT * FROM t1)) AS solo_en_t,
  (SELECT COUNT(*) FROM (SELECT * FROM t1 EXCEPT ALL SELECT * FROM t)) AS solo_en_t1

filas_t1,identicas,solo_en_t,solo_en_t1
49000,35159,14841,13841


In [15]:
%%sql
WITH solo_t AS (SELECT * FROM t EXCEPT ALL SELECT * FROM t1),
     solo_t1 AS (SELECT * FROM t1 EXCEPT ALL SELECT * FROM t)
SELECT
  COUNT(*) AS pares,
  COUNT(*) FILTER (WHERE a.amount IS DISTINCT FROM b.amount) AS cambia_amount,
  COUNT(*) FILTER (WHERE a.description IS DISTINCT FROM b.description) AS cambia_description,
  COUNT(*) FILTER (WHERE a.commercial_name IS DISTINCT FROM b.commercial_name) AS cambia_commercial
FROM solo_t a
JOIN solo_t1 b USING (id_cliente, date, product, fund, type)

pares,cambia_amount,cambia_description,cambia_commercial
3851,2350,1511,9


In [16]:
%%sql
SELECT type, COUNT(*) AS n
FROM t
GROUP BY type
ORDER BY n DESC

type,n
entrada,25423
salida,20631
IN,586
Entrada,547
ENTRADA,530
in,520
Salida,478
out,448
SALIDA,429
OUT,408


In [17]:
%%sql
SELECT type, COUNT(*) AS n
FROM t1
GROUP BY type
ORDER BY n DESC

type,n
entrada,24940
salida,20187
IN,549
Entrada,543
ENTRADA,537
in,531
Salida,457
out,455
SALIDA,407
OUT,394


In [18]:
%%sql
SELECT '[' || product || ']' AS product_visible,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1
FROM (SELECT product, 'T' AS origen FROM t
      UNION ALL
      SELECT product, 'T1' FROM t1)
GROUP BY product
ORDER BY product

product_visible,en_t,en_t1
[Acciones],6225,6079
[Bonos],6222,6169
[CDT],6333,6181
[Cuenta de Ahorro],6343,6221
[Divisas],6222,6107
[ETF],6201,6065
[Fondo de Inversión],6257,6095
[Fondo de Pensión],6197,6083


In [19]:
%%sql
SELECT regexp_replace(id_cliente, '\d', '9', 'g') AS patron,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1,
       MIN(id_cliente) AS ejemplo
FROM (SELECT id_cliente, 'T' AS origen FROM t
      UNION ALL
      SELECT id_cliente, 'T1' FROM t1)
GROUP BY patron
ORDER BY en_t DESC

patron,en_t,en_t1,ejemplo
CLI999999,50000,49000,CLI000001


In [20]:
%%sql
SELECT
  (SELECT COUNT(DISTINCT id_cliente) FROM t)  AS clientes_t,
  (SELECT COUNT(DISTINCT id_cliente) FROM t1) AS clientes_t1,
  (SELECT COUNT(*) FROM (SELECT id_cliente FROM t EXCEPT SELECT id_cliente FROM t1)) AS solo_en_t,
  (SELECT COUNT(*) FROM (SELECT id_cliente FROM t1 EXCEPT SELECT id_cliente FROM t)) AS solo_en_t1

clientes_t,clientes_t1,solo_en_t,solo_en_t1
3000,3000,0,0


In [21]:
%%sql
SELECT regexp_replace(date, '\d', '9', 'g') AS patron,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1,
       MIN(date) AS ejemplo_min,
       MAX(date) AS ejemplo_max
FROM (SELECT date, 'T' AS origen FROM t
      UNION ALL
      SELECT date, 'T1' FROM t1)
GROUP BY patron
ORDER BY en_t DESC

patron,en_t,en_t1,ejemplo_min,ejemplo_max
9999-99-99,46518,45567,2024-09-15,2024-10-16
99/99/9999,3482,3433,01/10/2024,30/09/2024


In [22]:
%%sql
SELECT
  MAX(CAST(split_part(date, '/', 1) AS INT)) AS max_primera_parte,
  MAX(CAST(split_part(date, '/', 2) AS INT)) AS max_segunda_parte,
  COUNT(*) AS filas
FROM (SELECT date FROM t UNION ALL SELECT date FROM t1)
WHERE date LIKE '%/%'

max_primera_parte,max_segunda_parte,filas
30,10,6915


In [23]:
%%sql
WITH fechas AS (
  SELECT date AS original,
         COALESCE(try_strptime(date, '%Y-%m-%d'), try_strptime(date, '%d/%m/%Y'))::DATE AS fecha
  FROM (SELECT date FROM t UNION ALL SELECT date FROM t1)
)
SELECT
  COUNT(*) FILTER (WHERE fecha IS NULL) AS no_convertibles,
  MIN(fecha) AS fecha_min,
  MAX(fecha) AS fecha_max
FROM fechas

no_convertibles,fecha_min,fecha_max
0,2024-09-15,2024-10-16


In [27]:
%%sql
SELECT '[' || fund || ']' AS fund_visible,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1
FROM (SELECT fund, 'T' AS origen FROM t
      UNION ALL
      SELECT fund, 'T1' FROM t1)
GROUP BY fund
ORDER BY LOWER(fund)

fund_visible,en_t,en_t1
[ crecimiento],366,352
[ Renta Fija],367,354
[ Renta Variable ],392,404
[BALANCEADO],398,398
[Balanceado],6254,6060
[balanceado],359,367
[conservador],377,366
[Conservador],6275,6177
[Conservador ],358,368
[Crecimiento],6352,6212


In [28]:
%%sql
SELECT '[' || LOWER(TRIM(fund)) || ']' AS fund_limpio,
       COUNT(*) AS n
FROM (SELECT fund FROM t UNION ALL SELECT fund FROM t1)
GROUP BY fund_limpio
ORDER BY fund_limpio

fund_limpio,n
[balanceado],13836
[conservador],13921
[crecimiento],14029
[internacional],14084
[mercado monetario],697
[mercado monetario],13175
[renta fija],14650
[renta variable],14608


In [29]:
%%sql
SELECT DISTINCT
  '[' || fund || ']' AS visible,
  LENGTH(fund) AS largo,
  encode(fund) AS bytes
FROM (SELECT fund FROM t UNION ALL SELECT fund FROM t1)
WHERE LOWER(fund) LIKE '%mercado%'

visible,largo,bytes
[mercado monetario],17,b'mercado monetario'
[Mercado Monetario],17,b'Mercado Monetario'
[Mercado Monetario],18,b'Mercado Monetario'


In [30]:
%%sql
SELECT DISTINCT
  LENGTH(fund) AS largo,
  unicode(right(fund, 1)) AS codigo_ultimo,
  unicode(left(fund, 1)) AS codigo_primero
FROM (SELECT fund FROM t UNION ALL SELECT fund FROM t1)
WHERE LOWER(fund) LIKE '%mercado%'


largo,codigo_ultimo,codigo_primero
18,111,77
17,111,77
17,111,109


In [31]:
%%sql
SELECT DISTINCT
  LENGTH(fund) AS largo,
  unicode(substr(fund, 8, 1)) AS pos_8,
  unicode(substr(fund, 9, 1)) AS pos_9,
  unicode(substr(fund, 10, 1)) AS pos_10
FROM (SELECT fund FROM t UNION ALL SELECT fund FROM t1)
WHERE LOWER(fund) LIKE '%mercado%'

largo,pos_8,pos_9,pos_10
17,32,109,111
18,32,32,77
17,32,77,111


In [32]:
%%sql
SELECT '[' || LOWER(TRIM(regexp_replace(fund, '\s+', ' ', 'g'))) || ']' AS fund_limpio,
       COUNT(*) AS n
FROM (SELECT fund FROM t UNION ALL SELECT fund FROM t1)
GROUP BY fund_limpio
ORDER BY fund_limpio

fund_limpio,n
[balanceado],13836
[conservador],13921
[crecimiento],14029
[internacional],14084
[mercado monetario],13872
[renta fija],14650
[renta variable],14608


In [33]:
%%sql
SELECT LOWER(TRIM(regexp_replace(description, '\s+', ' ', 'g'))) AS desc_limpia,
       COUNT(DISTINCT description) AS variantes_crudas,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1
FROM (SELECT description, 'T' AS origen FROM t
      UNION ALL
      SELECT description, 'T1' FROM t1)
GROUP BY desc_limpia
ORDER BY desc_limpia

desc_limpia,variantes_crudas,en_t,en_t1
ajuste por valoración,1,4514,4452
comisión de administración,1,4621,4488
compra de activo,1,4453,4322
depósito inicial,1,4474,4420
redención anticipada,1,4524,4497
reinversión automática,1,4574,4490
rendimiento del período,1,4499,4380
retiro parcial,1,4540,4429
traslado entre fondos,1,4605,4503
venta de activo,1,4633,4534


In [34]:
%%sql
SELECT description,
       COUNT(*) FILTER (WHERE LOWER(TRIM(type)) IN ('entrada','in')) AS entradas,
       COUNT(*) FILTER (WHERE LOWER(TRIM(type)) IN ('salida','out')) AS salidas
FROM (SELECT description, type FROM t UNION ALL SELECT description, type FROM t1)
GROUP BY description
ORDER BY description

description,entradas,salidas
Ajuste por valoración,4904,4062
Comisión de administración,4915,4194
Compra de activo,4836,3939
Depósito inicial,4852,4042
Redención anticipada,5041,3980
Reinversión automática,5054,4010
Rendimiento del período,4973,3906
Retiro parcial,4970,3999
Traslado entre fondos,5048,4060
Venta de activo,5070,4097


In [35]:
%%sql
SELECT description,
       COUNT(*) FILTER (WHERE LOWER(TRIM(type)) IN ('entrada','in')) AS entradas,
       COUNT(*) FILTER (WHERE LOWER(TRIM(type)) IN ('salida','out')) AS salidas
FROM (SELECT description, type FROM t UNION ALL SELECT description, type FROM t1)
GROUP BY description
ORDER BY description

description,entradas,salidas
Ajuste por valoración,4904,4062
Comisión de administración,4915,4194
Compra de activo,4836,3939
Depósito inicial,4852,4042
Redención anticipada,5041,3980
Reinversión automática,5054,4010
Rendimiento del período,4973,3906
Retiro parcial,4970,3999
Traslado entre fondos,5048,4060
Venta de activo,5070,4097


In [36]:
%%sql
SELECT '[' || commercial_name || ']' AS cn_visible,
       COUNT(*) FILTER (WHERE origen = 'T') AS en_t,
       COUNT(*) FILTER (WHERE origen = 'T1') AS en_t1
FROM (SELECT commercial_name, 'T' AS origen FROM t
      UNION ALL
      SELECT commercial_name, 'T1' FROM t1)
GROUP BY commercial_name
ORDER BY LOWER(commercial_name)

cn_visible,en_t,en_t1
[Bancolombia],4195,4071
[BBVA],4181,4071
[BTG Pactual],4066,3967
[Corficolombiana],4136,4061
[Davivienda],4208,4148
[Fiduciaria Bogotá],4081,3990
[Itaú],4150,4105
[Scotiabank],4282,4196
[Skandia],4158,4067
[Valores Bancolombia],4236,4157


In [37]:
%%sql
SELECT
  origen,
  COUNT(*) AS filas,
  COUNT(*) - COUNT(amount) AS nulos,
  COUNT(*) FILTER (WHERE amount < 0) AS negativos,
  COUNT(*) FILTER (WHERE amount = 0) AS ceros,
  MIN(amount) AS minimo,
  MAX(amount) AS maximo,
  ROUND(AVG(amount), 2) AS promedio,
  ROUND(MEDIAN(amount), 2) AS mediana,
  COUNT(*) FILTER (WHERE amount <> ROUND(amount, 2)) AS mas_de_2_decimales
FROM (SELECT amount, 'T' AS origen FROM t UNION ALL SELECT amount, 'T1' FROM t1)
GROUP BY origen

origen,filas,nulos,negativos,ceros,minimo,maximo,promedio,mediana,mas_de_2_decimales
T,50000,1543,1034,940,-49905610.23,49998466.14,23389951.27,23884861.99,0
T1,49000,1440,989,903,-49905610.23,49998466.14,23408449.16,23884058.63,0


In [38]:
%%sql
SELECT
  CASE WHEN LOWER(TRIM(type)) IN ('entrada','in') THEN 'entrada' ELSE 'salida' END AS type_limpio,
  COUNT(*) FILTER (WHERE amount < 0) AS negativos,
  COUNT(*) FILTER (WHERE amount > 0) AS positivos
FROM (SELECT type, amount FROM t UNION ALL SELECT type, amount FROM t1)
GROUP BY type_limpio

type_limpio,negativos,positivos
entrada,2023,50009
salida,0,42142


In [44]:
%%sql
SELECT *,
  CASE WHEN LOWER(TRIM(type)) IN ('entrada','in') THEN 'entrada' ELSE 'salida' END AS type_limpio
FROM  t
WHERE type_limpio = 'entrada' AND amount < 0


id_cliente,date,product,type,fund,amount,description,commercial_name,type_limpio
CLI000816,2024-09-27,Fondo de Inversión,entrada,Crecimiento,-34964254.7,None,Corficolombiana,entrada
CLI000608,2024-09-17,Fondo de Pensión,entrada,Renta Fija,-49533840.34,Comisión de administración,Valores Bancolombia,entrada
CLI001571,15/09/2024,Bonos,entrada,Renta Fija,-26552158.24,Compra de activo,None,entrada
CLI002979,2024-10-13,Divisas,entrada,Conservador,-32749023.47,None,BTG Pactual,entrada
CLI002236,2024-09-24,Acciones,entrada,Internacional,-47869625.06,Ajuste por valoración,Itaú,entrada
CLI000661,2024-10-05,CDT,entrada,Renta Variable,-31477391.7,Redención anticipada,Itaú,entrada
CLI000445,2024-09-15,ETF,entrada,Crecimiento,-30737793.73,Ajuste por valoración,BTG Pactual,entrada
CLI002580,2024-09-18,Fondo de Pensión,entrada,Mercado Monetario,-12886483.56,Venta de activo,Fiduciaria Bogotá,entrada
CLI002818,2024-10-14,Divisas,entrada,Internacional,-11561538.46,Traslado entre fondos,BBVA,entrada
CLI000880,2024-10-12,Cuenta de Ahorro,entrada,Crecimiento,-7736584.6,Traslado entre fondos,Itaú,entrada
